In [1]:
from utils import *
#import episcanpy.api as epi
import time
import umap

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata

# Read data

In [2]:
#atac = sc.read_h5ad("../../h5ad_files/c_atac_spatial_peaks.h5ad")
#expr = sc.read_h5ad("../../h5ad_files/c_sp_trans_wide_ad.h5ad")

In [70]:
atac

AnnData object with n_obs × n_vars = 5274 × 72148
    obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'uniform_density', 'rna_count_based_density'
    var: 'chrom', 'start', 'end', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'is_training'
    uns: 'files', 'gearyC', 'spatial_neighbors'
    obsm: 'spatial'
    obsp: 'spatial_connectivities', 'spatial_distances'

In [3]:
'''
sc.pp.normalize_total(expr, target_sum=1e4)
sc.pp.log1p(expr)

#count_mat = atac.X.todense().T
count_mat = atac.X.T
tf_mat = 1.0 * count_mat / np.tile(np.sum(count_mat,axis=0), (count_mat.shape[0],1))
ATAC_count = np.log(1 + np.multiply(1e4*tf_mat,  np.tile((1.0 * count_mat.shape[1] / np.sum(count_mat,axis=1)).reshape(-1,1), (1,count_mat.shape[1]))))
atac.X = scipy.sparse.csc_matrix(ATAC_count.T)

atac, expr
'''

'\nsc.pp.normalize_total(expr, target_sum=1e4)\nsc.pp.log1p(expr)\n\n#count_mat = atac.X.todense().T\ncount_mat = atac.X.T\ntf_mat = 1.0 * count_mat / np.tile(np.sum(count_mat,axis=0), (count_mat.shape[0],1))\nATAC_count = np.log(1 + np.multiply(1e4*tf_mat,  np.tile((1.0 * count_mat.shape[1] / np.sum(count_mat,axis=1)).reshape(-1,1), (1,count_mat.shape[1]))))\natac.X = scipy.sparse.csc_matrix(ATAC_count.T)\n\natac, expr\n'

In [4]:
#atac.write_h5ad("desc_normalized_atac_peaks.h5ad")

In [5]:
#expr.write_h5ad("desc_normalized_rna.h5ad")

In [6]:
atac = sc.read_h5ad("desc_normalized_atac_peaks.h5ad")
expr = sc.read_h5ad("desc_normalized_rna.h5ad")

In [7]:
#filtered_atac = atac[:, atac.var['n_cells_by_counts'] > 10].copy()

In [8]:
#atac = filtered_atac

In [9]:
atac = atac[:, atac.var['n_cells_by_counts'] > 10].copy()

In [10]:
np.all(atac.obs_names == expr.obs_names)

True

# Run Descart

In [11]:
save_path = 'result/first_descart_test'
if not os.path.exists(save_path):
    os.makedirs(save_path)
seed_base = 1
tf = None
pc = 10
k = 20
similarity = 'cosine'
iter_time = 4
spmethod = 'threshold'
neighbor = 5
sp_dist = 'recip'
pre_select = 'highest'
peaks_num = 50000
distance = 'euclidean'
r = 0.4

num_select_peak = 20000
idx_atac, _, _, _, _, _,_, _ = run_descart(atac, num_select_peak, seed_base=seed_base, tfidf=tf, ifPCA=True, pc=pc, k=k, similarity=similarity, iters=iter_time, spmethod=spmethod,neighbor=neighbor,sp_dist=sp_dist, pre_select=pre_select, peaks_num=peaks_num, distance=distance,r=r)

num_select_peak = 2000
idx_expr, _, _, _, _, _,_, _ = run_descart(expr, num_select_peak, seed_base=seed_base, tfidf=tf, ifPCA=True, pc=pc, k=k, similarity=similarity, iters=iter_time, spmethod=spmethod,neighbor=neighbor,sp_dist=sp_dist, pre_select=pre_select, peaks_num=peaks_num, distance=distance,r=r)

idx_atac.shape, idx_expr.shape

AnnData object with n_obs × n_vars = 5274 × 72148
    obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'uniform_density', 'rna_count_based_density'
    var: 'chrom', 'start', 'end', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'is_training'
    uns: 'files', 'gearyC', 'spatial_neighbors'
    obsm: 'spatial'
    obsp: 'spatial_connectivities', 'spatial_distances'
(5274, 72148)
(5274, 5274)
(5274, 10)
0.0
compute scores


/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:258: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:277: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(


scores:
[10381.67716524 10381.83834126 10381.84893942 ... 30398.77289027
 32907.72012353 36544.23266873]
(5274, 5274)
(5274, 10)
0.0
compute scores


/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:258: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:277: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(


scores:
[10651.48909828 10651.61825198 10651.64862071 ... 30298.22920851
 32899.68127873 36349.29523418]
(5274, 5274)
(5274, 10)
0.0
compute scores


...

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:258: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:277: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(


scores:
[18855.09643351 18856.00791133 18857.67338159 ... 56134.28432648
 56555.1036081  56766.94185052]
(5274, 5274)
(5274, 10)
0.0
compute scores


/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:258: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:277: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(


scores:
[18855.06883036 18856.50122943 18857.74214495 ... 56134.16238706
 56555.71884035 56766.94559768]


((20000,), (2000,))

In [12]:
idx_atac, idx_expr

(array([18226,  8513, 51037, ..., 46390, 26010, 71830]),
 array([13881, 16527, 11934, ..., 13744, 13992,  4232]))

In [13]:
expr[:, 13881].var

,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,n_cells,highly_variable,means,dispersions,dispersions_norm,is_training
ptgs1,37,0.01154,0.011474,98.876404,38,3.663562,37,False,0.009383,0.569499,0.347026,False


In [14]:
atac[:, 65324].var

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training
chr17:80982326-80982825,chr17,80982326,80982825,40,0.013362,98.785302,44.0,40,False


# Gene-peak interaction detection

In [15]:
'''

file_path = '../data/'
dataset = 'E13_5-S1'
atac = sc.read_h5ad(file_path + dataset + '_atac' + '.h5ad')
atac.X = scipy.sparse.csc_matrix(atac.X)
atac.obs['label'] = atac.obs['Annotation_for_Combined']
 
expr = sc.read_h5ad(file_path + dataset + '_expr' + '.h5ad')
expr.X = scipy.sparse.csc_matrix(expr.X)
expr.obs['label'] = expr.obs['Annotation_for_Combined']
expr.var_names_make_unique()

barcode_set1 = set(atac.obs_names)
barcode_set2 = set(expr.obs_names)

common_barcodes = barcode_set1.intersection(barcode_set2)

atac = atac[atac.obs_names.isin(common_barcodes), :]
expr = expr[expr.obs_names.isin(common_barcodes), :]
expr = expr[atac.obs_names]

'''

atac_20000 = atac[:,idx_atac]
expr_2000 = expr[:,idx_expr]
atac_20000,expr_2000

(View of AnnData object with n_obs × n_vars = 5274 × 20000
     obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'uniform_density', 'rna_count_based_density'
     var: 'chrom', 'start', 'end', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'is_training'
     uns: 'files', 'gearyC', 'spatial_neighbors'
     obsm: 'spatial'
     obsp: 'spatial_connectivities', 'spatial_distances',
 View of AnnData object with n_obs × n_vars = 5274 × 2000
     obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'un

In [16]:
sc.pp.normalize_total(expr_2000, target_sum=10000)
sc.pp.log1p(expr_2000)
sc.pp.scale(expr_2000)
expr_2000

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/scanpy/preprocessing/_normalization.py:169: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


AnnData object with n_obs × n_vars = 5274 × 2000
    obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'uniform_density', 'rna_count_based_density', 'squidpy_domains', 'leiden'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'is_training', 'mean', 'std'
    uns: 'gearyC', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'overlap_genes', 'pca', 'spatial_neighbors', 'squidpy_domains_colors', 'subclass_colors', 'training_genes', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial', 'tangram_ct_pred'
    varm: 'PCs'
    obsp: 'connectivities', 'distances', 'spatial_connectivities', 'spatial_distances'

In [17]:
count_mat = atac_20000.X.toarray().T

In [18]:
tf_mat = 1.0 * count_mat / np.tile(np.sum(count_mat,axis=0), (count_mat.shape[0],1))

In [19]:
ATAC_count = np.log(1 + np.multiply(1e4*tf_mat,  np.tile((1.0 * count_mat.shape[1] / np.sum(count_mat,axis=1)).reshape(-1,1), (1,count_mat.shape[1]))))

In [20]:
atac_20000.X = scipy.sparse.csc_matrix(ATAC_count.T)

In [21]:
adata = anndata.concat([atac_20000, expr_2000], axis=1)
adata.obs['label'] = list(atac_20000.obs['label'])
adata.obsm['spatial'] = atac_20000.obsm['spatial']
adata

AnnData object with n_obs × n_vars = 5274 × 22000
    obs: 'label'
    var: 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'is_training'
    obsm: 'spatial'

In [22]:
adata.X = scipy.sparse.csc_matrix(adata.X)

ATAC_count_test = scale(adata.X.toarray())
count = ATAC_count_test.copy()
print(count.shape)
    
similarity_matrix_acb = np.zeros([count.shape[0], count.shape[0]])
similarity_matrix_spatial = np.zeros([count.shape[0], count.shape[0]])

distance_matrix = np.zeros([count.shape[0], count.shape[0]])
print(distance_matrix.shape)

count = PCA(n_components=pc,random_state=int(seed_base*1000)).fit_transform(count)
print(count.shape)
if similarity == 'Jaccard':
    if distance == 'euclidean':
        diff = count[:, np.newaxis, :] - count[np.newaxis, :, :]
        distance_matrix = np.linalg.norm(diff, axis=2)
        np.fill_diagonal(distance_matrix, np.inf)
    elif distance == 'cosine':
        count_norm = np.linalg.norm(count, axis=1, keepdims=True)
        dot_product_matrix = np.dot(count, count.T)
        distance_matrix = 1 - dot_product_matrix / (count_norm * count_norm.T)
        np.fill_diagonal(distance_matrix, np.inf)

distance_matrix_acb = np.copy(distance_matrix)

spatial_data = adata.obsm['spatial']
diff = spatial_data[:, np.newaxis, :] - spatial_data[np.newaxis, :, :]
distance_matrix = np.linalg.norm(diff, axis=2)
np.fill_diagonal(distance_matrix, np.inf)
distance_matrix_spatial = np.copy(distance_matrix)

similarity_matrix_acb = np.zeros([count.shape[0], count.shape[0]])
similarity_matrix_spatial = np.zeros([count.shape[0], count.shape[0]])
if spmethod == 'threshold':
    min_dist = np.min(distance_matrix_spatial)
    if sp_dist == 'const':
        similarity_matrix_spatial = np.array(distance_matrix_spatial <= neighbor * min_dist,dtype=int)
    elif sp_dist == 'recip':
        similarity_matrix_spatial = min_dist / distance_matrix_spatial
        similarity_matrix_spatial = similarity_matrix_spatial * (distance_matrix_spatial <= neighbor * min_dist)
    else:
        similarity_matrix_spatial = (min_dist / distance_matrix_spatial)**2
        similarity_matrix_spatial = similarity_matrix_spatial * (distance_matrix_spatial <= neighbor * min_dist)

elif spmethod == 'SNN':
    neighbor_index = np.argsort(distance_matrix_spatial, axis=1)[:,0:k]
    if similarity == "Jaccard":
        for i in range(count.shape[0]):
            for j in range(i):
                intersect_num = len(np.intersect1d(neighbor_index[i,:], neighbor_index[j,:]))
                similarity_matrix_spatial[i][j] = 1.0*intersect_num/(2*k-intersect_num)
                similarity_matrix_spatial[j][i] = 1.0*intersect_num/(2*k-intersect_num)
if similarity == "Jaccard":
    neighbor_index = np.argsort(distance_matrix_acb, axis=1)[:,0:k]
    for i in range(count.shape[0]):
        for j in range(i):
            intersect_num = len(np.intersect1d(neighbor_index[i,:], neighbor_index[j,:]))
            similarity_matrix_acb[i][j] = 1.0*intersect_num/(2*k-intersect_num)
            similarity_matrix_acb[j][i] = 1.0*intersect_num/(2*k-intersect_num)
elif similarity == "cosine":
    similarity_matrix_acb = sklearn.metrics.pairwise.cosine_similarity(count)
    for i in range(count.shape[0]):
        similarity_matrix_acb[i][i] = -float('inf')
    neighbor_index = np.argsort(similarity_matrix_acb, axis=1)[:,0:(count.shape[0]-k)]
    for i in range(count.shape[0]):
        similarity_matrix_acb[i,neighbor_index[i,:]] = 0
    print(similarity_matrix_acb[0,0])

similarity_matrix = (1 - r) * similarity_matrix_acb + r*similarity_matrix_spatial


print('compute scores')
scores = np.zeros(adata.n_vars)
X_processed = ATAC_count_test

temp_matrix = np.matmul(similarity_matrix, X_processed)
scores = np.sum(X_processed * temp_matrix, axis=0)
sorted_index = np.argsort(scores)
idx = sorted_index[::-1]
print('scores:')
print(scores[idx])

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:258: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:277: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(


(5274, 22000)
(5274, 5274)
(5274, 10)
0.0
compute scores
scores:
[56989.68066684 56104.62258559 55379.86121635 ...  5494.15635106
  4765.54666741  4687.7243395 ]


In [23]:
start_time = time.time()
pdist_ = peak_modules_(selected_peaks_data=X_processed, similarity_matrix=similarity_matrix, method='complete')
pdist_ = (pdist_+pdist_.T)/2
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Running time: {elapsed_time:.6f} seconds")
print("Peak modules calculation has been done.")

pdist_.shape

Running time: 250.915735 seconds
Peak modules calculation has been done.


(22000, 22000)

In [24]:
pdist

<function scipy.spatial.distance.pdist(X, metric='euclidean', *, out=None, **kwargs)>

In [25]:
g_p_similarity = pdist_[20000:,:20000]

## The most likely gene and peak

In [27]:
index = np.unravel_index(np.argmax(g_p_similarity, axis=None), g_p_similarity.shape)
index

(1935, 19998)

In [28]:
expr_2000[:,1935].var

,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,n_cells,highly_variable,means,dispersions,dispersions_norm,is_training,mean,std
gramd3,124,0.048284,0.047155,96.234437,159,5.075174,123,True,0.057365,1.155377,1.361016,False,0.632511,0.596935


In [29]:
atac_20000[:,19998].var

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training
chr6:85906858-85907357,chr6,85906858,85907357,16,0.004859,99.514121,16.0,16,False


gramd3

chr6:85906858-85907357

In [30]:
# Find the index of the second largest value in the flattened array
# np.argsort(g_p_similarity, axis=None) returns the indices that would sort the flattened array.
# [-2] gets the second-to-last index, which corresponds to the second largest value's position
second_max_flat_index = np.argsort(g_p_similarity, axis=None)[-2]

# Convert the flat index to multi-dimensional indices using the original shape
second_index = np.unravel_index(second_max_flat_index, g_p_similarity.shape)

print(f"The indices of the largest entry are: {index}")
print(f"The indices of the second largest entry are: {second_index}")

The indices of the largest entry are: (1935, 19998)
The indices of the second largest entry are: (1667, 19998)


In [31]:
expr_2000[:,1667].var

,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,n_cells,highly_variable,means,dispersions,dispersions_norm,is_training,mean,std
csrp1,85,0.02976,0.029326,97.418767,98,4.59512,85,True,0.037922,1.444277,1.861021,False,0.480593,0.524611


In [32]:
atac_20000[:,19998].var

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training
chr6:85906858-85907357,chr6,85906858,85907357,16,0.004859,99.514121,16.0,16,False


### examining rorb

In [33]:
def get_top_correlated_peaks_on_chromosome(
    gene_name: str,
    chromosome: str,
    expr: anndata.AnnData,
    atac: anndata.AnnData,
    g_p_similarity: np.ndarray,
    top_n: int = 100
) -> pd.DataFrame:
    """
    Retrieve top correlated peaks with a gene restricted to a specific chromosome.

    Parameters:
    - gene_name (str): Name of the gene of interest.
    - chromosome (str): Target chromosome, e.g., "chr19".
    - expr (anndata.AnnData): Tangram-mapped expression data (used for gene index).
    - atac (anndata.AnnData): Tangram-mapped ATAC data with 'chrom' in var.
    - g_p_similarity (np.ndarray): Gene-peak correlation matrix (genes x peaks).
    - top_n (int): Number of top correlated peaks to consider.

    Returns:
    - pd.DataFrame: Peaks on the given chromosome among the top N most correlated.
    """
    if gene_name not in expr.var_names:
        raise ValueError(f"Gene {gene_name} not found in expr.var_names.")

    gene_idx = list(expr.var_names).index(gene_name)
    peak_scores = g_p_similarity[gene_idx, :]
    top_peak_indices = np.argsort(peak_scores)[::-1][:top_n]

    top_peaks = atac.var.iloc[top_peak_indices]
    top_peaks_on_chr = top_peaks[top_peaks["chrom"] == chromosome]

    return top_peaks_on_chr

In [34]:
get_top_correlated_peaks_on_chromosome(
    gene_name="rorb",
    chromosome="chr19",
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    top_n=100
)

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training
chr19:18972253-18972752,chr19,18972253,18972752,11,0.003340,99.665958,11.0,11,False
chr19:28679927-28680426,chr19,28679927,28680426,14,0.004251,99.574856,14.0,14,False


In [35]:
get_top_correlated_peaks_on_chromosome(
    gene_name="tox",
    chromosome="chr4",
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    top_n=100
)

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training
chr4:141865001-141865500,chr4,141865001,141865500,12,0.003644,99.635591,12.0,12,False
chr4:47440904-47441403,chr4,47440904,47441403,12,0.003644,99.635591,12.0,12,False


In [ ]:
g_p_similarity[gene_idx, top_peaks]

In [40]:
 pd = get_top_correlated_peaks_on_chromosome(
    gene_name="pdzrn3",
    chromosome="chr6",
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    top_n=100
)

In [41]:
pd["dis"] = (pd["start"]+250) - 101240714

In [42]:
pd

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training,dis
chr6:39849654-39850153,chr6,39849654,39850153,11,0.003340,99.665958,11.0,11,False,-61390810
chr6:148276284-148276783,chr6,148276284,148276783,15,0.004555,99.544488,15.0,15,False,47035820
chr6:103927053-103927552,chr6,103927053,103927552,11,0.003644,99.665958,12.0,11,False,2686589
chr6:31049212-31049711,chr6,31049212,31049711,18,0.005770,99.453386,19.0,18,False,-70191252
chr6:56913048-56913547,chr6,56913048,56913547,15,0.004555,99.544488,15.0,15,False,-44327416
chr6:20574577-20575076,chr6,20574577,20575076,12,0.003948,99.635591,13.0,12,False,-80665887
chr6:64596180-64596679,chr6,64596180,64596679,15,0.004555,99.544488,15.0,14,False,-36644284


In [49]:
rspo=get_top_correlated_peaks_on_chromosome(
    gene_name="rspo1",
    chromosome="chr4",
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    top_n=100
)

rspo1 Chr4:124880223-124902892 -> 124882892

In [50]:
rspo["dis"] = (rspo["start"]+250) - 124882892

In [51]:
rspo

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training,dis
chr4:65833161-65833660,chr4,65833161,65833660,13,0.003948,99.605223,13.0,13,False,-59049481
chr4:63489659-63490158,chr4,63489659,63490158,11,0.003340,99.665958,11.0,11,False,-61392983
chr4:53554081-53554580,chr4,53554081,53554580,14,0.004251,99.574856,14.0,14,False,-71328561
chr4:145104934-145105433,chr4,145104934,145105433,22,0.006985,99.331916,23.0,22,False,20222292


In [66]:
cux=get_top_correlated_peaks_on_chromosome(
    gene_name="cux2",
    chromosome="chr18",
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    top_n=100
)

In [69]:
cux

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training,dis
chr18:83766330-83766829,chr18,83766330,83766829,14,0.004251,99.574856,14.0,14,False,-38321942
chr18:30753184-30753683,chr18,30753184,30753683,14,0.004251,99.574856,14.0,14,False,-91335088
chr18:67650853-67651352,chr18,67650853,67651352,12,0.003644,99.635591,12.0,12,False,-54437419
chr18:41697389-41697888,chr18,41697389,41697888,12,0.003644,99.635591,12.0,12,False,-80390883
chr18:63975353-63975852,chr18,63975353,63975852,12,0.004251,99.635591,14.0,12,False,-58112919
chr18:22358237-22358736,chr18,22358237,22358736,17,0.005162,99.483753,17.0,17,False,-99730035
chr18:64972114-64972613,chr18,64972114,64972613,14,0.004251,99.574856,14.0,14,False,-57116158
chr18:58281836-58282335,chr18,58281836,58282335,21,0.006377,99.362284,21.0,21,False,-63806436


Chr5:121996025-122188522.  122088522

In [68]:
cux["dis"] = (cux["start"]+250) - 122088522

In [58]:
cux

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training,dis
chr5:38775445-38775944,chr5,38775445,38775944,23,0.006985,99.301549,23.0,23,False,-83312827
chr5:115347527-115348026,chr5,115347527,115348026,12,0.003644,99.635591,12.0,12,False,-6740745
chr5:136476199-136476698,chr5,136476199,136476698,13,0.003948,99.605223,13.0,13,False,14387927
chr5:18274558-18275057,chr5,18274558,18275057,13,0.004251,99.605223,14.0,13,False,-103813714


lamp5

In [61]:
lam=get_top_correlated_peaks_on_chromosome(
    gene_name="lamp5",
    chromosome="chr2",
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    top_n=100
)

In [64]:
135902837
lam["dis"] = (lam["start"]+250) - 135902837

### chr2:135970957-135971456

In [65]:
lam

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training,dis
chr2:135970957-135971456,chr2,135970957,135971456,13,0.003948,99.605223,13.0,13,False,68370
chr2:28303876-28304375,chr2,28303876,28304375,15,0.004859,99.544488,16.0,15,False,-107598711
chr2:136402119-136402618,chr2,136402119,136402618,13,0.004251,99.605223,14.0,13,False,499532
chr2:104497642-104498141,chr2,104497642,104498141,19,0.005770,99.423019,19.0,19,False,-31404945
chr2:84549632-84550131,chr2,84549632,84550131,11,0.003340,99.665958,11.0,11,False,-51352955
chr2:178329501-178330000,chr2,178329501,178330000,13,0.003948,99.605223,13.0,13,False,42426914
chr2:75151468-75151967,chr2,75151468,75151967,13,0.003948,99.605223,13.0,13,False,-60751119


fam19a2 Chr10:123099901-123577109 -> 123349901

In [71]:
fam=get_top_correlated_peaks_on_chromosome(
    gene_name="fam19a2",
    chromosome="chr10",
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    top_n=100
)

In [73]:
123349901
fam["dis"] = (fam["start"]+250) - 123349901

In [74]:
fam

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training,dis
chr10:30566336-30566835,chr10,30566336,30566835,12,0.003644,99.635591,12.0,12,False,-92783315
chr10:9892608-9893107,chr10,9892608,9893107,13,0.003948,99.605223,13.0,13,False,-113457043
chr10:99553014-99553513,chr10,99553014,99553513,11,0.003340,99.665958,11.0,10,False,-23796637
chr10:90770206-90770705,chr10,90770206,90770705,11,0.003948,99.665958,13.0,11,False,-32579445
chr10:68165220-68165719,chr10,68165220,68165719,27,0.008503,99.180079,28.0,27,False,-55184431


sulf1 Chr1:12762501-12931416  -> 12842501

In [75]:
sul=get_top_correlated_peaks_on_chromosome(
    gene_name="sulf1",
    chromosome="chr1",
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    top_n=100
)

In [76]:
sul["dis"] = (sul["start"]+250) - 12842501

In [77]:
sul

,chrom,start,end,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells,is_training,dis
chr1:165875561-165876060,chr1,165875561,165876060,11,0.003340,99.665958,11.0,11,False,153033310
chr1:21275419-21275918,chr1,21275419,21275918,12,0.003644,99.635591,12.0,12,False,8433168
chr1:66384955-66385454,chr1,66384955,66385454,12,0.003644,99.635591,12.0,12,False,53542704
chr1:182816656-182817155,chr1,182816656,182817155,12,0.004251,99.635591,14.0,12,False,169974405
chr1:5249453-5249952,chr1,5249453,5249952,14,0.004251,99.574856,14.0,14,False,-7592798


In [ ]:
accs = expr_2000.X[:,index[0]]
coord_x = np.array(atac_20000.obsm['spatial'][:,0])
coord_y = np.array(atac_20000.obsm['spatial'][:,1])
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()

accs = scale(atac_20000.X.toarray())[:,index[1]]
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()

## Gene to peak

In [ ]:
expr_matrix = expr_2000.X
atac_matrix = scale(atac_20000.X.toarray())
expr_matrix.shape, atac_matrix.shape

In [ ]:
a = anndata.AnnData(scipy.sparse.csc_matrix(atac_matrix))
a.obs_names = list(atac.obs_names)
a.obs['label'] = list(atac.obs['label'])
a.obsm['spatial'] = atac.obsm['spatial']

b = anndata.AnnData(scipy.sparse.csc_matrix(expr_matrix))
b.obs_names = list(expr.obs_names)
b.obs['label'] = list(expr.obs['label'])
b.obsm['spatial'] = expr.obsm['spatial']

result = np.zeros_like(g_p_similarity)
for col in range(g_p_similarity.shape[1]):
    indices = np.argpartition(g_p_similarity[:, col], -20)[-20:]
    result[indices, col] = g_p_similarity[indices, col]
    
    indices = np.argpartition(g_p_similarity[:, col],5)[:5]
    result[indices, col] = g_p_similarity[indices, col]
result.shape

In [ ]:
atac_matrix_pred = expr_matrix @ result
atac_matrix_pred = scale(atac_matrix_pred)
atac_matrix_pred.shape

In [ ]:
accs = atac_matrix[:,19998]
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()

accs = atac_matrix_pred[:,19998]
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()

## Peak to gene

In [ ]:
g_p_similarity_T = g_p_similarity.T
result = np.zeros_like(g_p_similarity_T)
for col in range(g_p_similarity_T.shape[1]):
    indices = np.argpartition(g_p_similarity_T[:, col], -20)[-20:]
    result[indices, col] = g_p_similarity_T[indices, col]
    
    indices = np.argpartition(g_p_similarity_T[:, col],5)[:5]
    result[indices, col] = g_p_similarity_T[indices, col]
result.shape

In [ ]:
expr_matrix_pred = atac_matrix @ result
expr_matrix_pred = scale(expr_matrix_pred)
expr_matrix_pred.shape

In [ ]:
accs = expr_matrix[:,1999]
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()

accs = expr_matrix_pred[:,1999]
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()